# Dim Store Promotions - Silver Build & Reconciliation

1. Rebuilds `uc2_silver_dim_store` as a 3-way fan-in
   (`uc1_raw_store_transactions` PRIMARY, `uc1_raw_inventory_movements`
   SECONDARY, `uc1_raw_store_promotions` TERTIARY), adding
   `promo_count` / `avg_discount_pct` / `last_promo_end_date`.
2. Builds the new standalone `uc2_silver_dim_store_promotions` dim table,
   fed solely by `uc1_raw_store_promotions`.
3. Registers both in `uc2_layer_registry_b` and computes
   `JOIN_MATCH_RATE` / `JOIN_ORPHAN_RATE` reconciliation results for the
   new TERTIARY source into `uc2_reconciliation_results_b`.

Depends on `agent1/data_generation/05_generate_store_promotions` having
already populated `uc1_raw_store_promotions`.

In [0]:
dbutils.widgets.text("catalog", "learning_dev", "Catalog")
dbutils.widgets.text("schema", "s819534", "Schema")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

TXN_TABLE = f"{catalog}.{schema}.uc1_raw_store_transactions"
INV_TABLE = f"{catalog}.{schema}.uc1_raw_inventory_movements"
PROMO_TABLE = f"{catalog}.{schema}.uc1_raw_store_promotions"
DIM_STORE_TABLE = f"{catalog}.{schema}.uc2_silver_dim_store"
DIM_STORE_PROMO_TABLE = f"{catalog}.{schema}.uc2_silver_dim_store_promotions"
LAYER_REGISTRY = f"{catalog}.{schema}.uc2_layer_registry_b"
RECON_RESULTS = f"{catalog}.{schema}.uc2_reconciliation_results_b"

## 1. New standalone dim table: `uc2_silver_dim_store_promotions`

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {DIM_STORE_PROMO_TABLE} AS
SELECT
  store_id,
  COUNT(*) AS promo_count,
  ROUND(AVG(discount_pct), 2) AS avg_discount_pct,
  MAX(end_date) AS last_promo_end_date,
  current_timestamp() AS dim_updated_timestamp
FROM {PROMO_TABLE}
WHERE store_id IS NOT NULL AND store_id != ''
GROUP BY store_id
""")

display(spark.table(DIM_STORE_PROMO_TABLE))

## 2. Rebuild `uc2_silver_dim_store` as a 3-way fan-in

PRIMARY (transactions) and SECONDARY (inventory) columns are reproduced
unchanged; TERTIARY (promotions) columns are added via a LEFT JOIN on
`store_id` against the dim table built above.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {DIM_STORE_TABLE} AS
SELECT
  t.store_id,
  t.store_region,
  t.last_transaction_date,
  t.transaction_count,
  t.total_net_amount,
  m.last_movement_date,
  m.movement_count,
  m.total_cost_impact,
  p.promo_count,
  p.avg_discount_pct,
  p.last_promo_end_date,
  current_timestamp() AS dim_updated_timestamp
FROM (
  SELECT store_id, store_region, MAX(transaction_date) AS last_transaction_date,
         COUNT(*) AS transaction_count, SUM(net_amount) AS total_net_amount
  FROM {TXN_TABLE}
  WHERE store_id IS NOT NULL AND store_id != ''
  GROUP BY store_id, store_region
) t
LEFT JOIN (
  SELECT store_id, MAX(business_date) AS last_movement_date,
         COUNT(*) AS movement_count, SUM(total_cost_impact) AS total_cost_impact
  FROM {INV_TABLE}
  WHERE store_id IS NOT NULL AND store_id != ''
  GROUP BY store_id
) m ON t.store_id = m.store_id
LEFT JOIN {DIM_STORE_PROMO_TABLE} p ON t.store_id = p.store_id
""")

display(spark.table(DIM_STORE_TABLE))

## 3. Layer registry - lineage metadata

Idempotent: deletes any prior rows for these layer_ids before inserting,
so the notebook can be safely re-run.

In [0]:
registry_rows = [
    ("lyr-promo-raw", PROMO_TABLE, "BRONZE", PROMO_TABLE,
     catalog, schema, "uc1_raw_store_promotions", "business_date",
     1.0, 120, '["discount_pct"]', True, "agent2_admin", None, None, None, None),
    ("lyr-dim-store-promo", PROMO_TABLE, "SILVER", DIM_STORE_TABLE,
     catalog, schema, "uc2_silver_dim_store", "last_promo_end_date",
     None, 60, '["promo_count", "avg_discount_pct"]', True, "agent2_admin",
     "store_id", "LEFT", "TERTIARY", 0.95),
    ("lyr-dim-store-promo-dim", PROMO_TABLE, "SILVER", DIM_STORE_PROMO_TABLE,
     catalog, schema, "uc2_silver_dim_store_promotions", "last_promo_end_date",
     1.0, 60, '["promo_count", "avg_discount_pct"]', True, "agent2_admin",
     None, None, None, None),
]

layer_ids = "', '".join(r[0] for r in registry_rows)
spark.sql(f"DELETE FROM {LAYER_REGISTRY} WHERE layer_id IN ('{layer_ids}')")

from pyspark.sql import Row

registry_df = spark.createDataFrame(
    [
        Row(
            layer_id=r[0], source_fully_qualified_name=r[1], layer_name=r[2],
            layer_fully_qualified_name=r[3], layer_catalog=r[4], layer_schema=r[5],
            layer_table=r[6], business_date_column=r[7], expected_survival_rate=r[8],
            expected_latency_minutes=r[9], reconciliation_metrics=r[10], is_active=r[11],
            registered_by=r[12], join_key_column=r[13], join_type=r[14],
            source_role=r[15], expected_match_rate=r[16],
        )
        for r in registry_rows
    ]
).withColumn("registered_timestamp", __import__("pyspark").sql.functions.current_timestamp())

registry_df.write.mode("append").saveAsTable(LAYER_REGISTRY)

display(spark.sql(f"SELECT * FROM {LAYER_REGISTRY} WHERE layer_id IN ('{layer_ids}')"))

## 4. Reconciliation: JOIN_MATCH_RATE / JOIN_ORPHAN_RATE for the new TERTIARY source

In [0]:
import uuid
import json

match_stats = spark.sql(f"""
SELECT
  (SELECT COUNT(DISTINCT store_id) FROM {PROMO_TABLE} WHERE store_id IS NOT NULL AND store_id != '') AS promo_keys,
  (SELECT COUNT(*) FROM {DIM_STORE_TABLE}) AS target_rows,
  (SELECT COUNT(*) FROM {DIM_STORE_TABLE} d LEFT JOIN {PROMO_TABLE} p ON d.store_id = p.store_id WHERE p.store_id IS NULL) AS orphaned_target_rows
""").collect()[0]

promo_keys = match_stats["promo_keys"]
target_rows = match_stats["target_rows"]
orphaned_target_rows = match_stats["orphaned_target_rows"]
matched_keys = target_rows - orphaned_target_rows
match_rate = matched_keys / target_rows if target_rows else 0.0
expected_match_rate = 0.95
deviation_pct = round(abs(match_rate - expected_match_rate) / expected_match_rate * 100, 2)
match_status = "PASS" if match_rate >= expected_match_rate else "FAIL"

orphan_rate = orphaned_target_rows / target_rows if target_rows else 0.0
orphan_status = "PASS" if orphan_rate <= 0.05 else "FAIL"

run_id = str(uuid.uuid4())

match_detail = json.dumps({
    "source_keys": promo_keys,
    "matched_keys": matched_keys,
    "match_rate": match_rate,
    "expected_match_rate": expected_match_rate,
    "target_table": DIM_STORE_TABLE,
    "join_key": "store_id",
})

orphan_detail = json.dumps({
    "target_table": DIM_STORE_TABLE,
    "total_target_rows": target_rows,
    "orphaned_from_txn": 0,
    "orphaned_from_inventory": 0,
    "orphaned_from_promotions": orphaned_target_rows,
})

recon_rows = [
    Row(
        result_id=str(uuid.uuid4()), reconciliation_run_id=run_id,
        source_fully_qualified_name=PROMO_TABLE, from_layer="BRONZE", to_layer="SILVER",
        check_type="JOIN_MATCH_RATE", metric_name="store_id_match_rate", check_status=match_status,
        source_value=float(promo_keys), target_value=float(matched_keys),
        expected_value=expected_match_rate, deviation_percentage=deviation_pct,
        threshold_warning=5.0, threshold_critical=15.0, check_detail=match_detail,
        agent1_overall_status=match_status,
    ),
    Row(
        result_id=str(uuid.uuid4()), reconciliation_run_id=run_id,
        source_fully_qualified_name=PROMO_TABLE, from_layer="SILVER", to_layer="SILVER",
        check_type="JOIN_ORPHAN_RATE", metric_name="dim_store_orphan_rate", check_status=orphan_status,
        source_value=float(target_rows), target_value=float(orphaned_target_rows),
        expected_value=0.0, deviation_percentage=round(orphan_rate * 100, 2),
        threshold_warning=5.0, threshold_critical=15.0, check_detail=orphan_detail,
        agent1_overall_status=orphan_status,
    ),
]

from pyspark.sql.functions import current_date, current_timestamp, lit

recon_df = spark.createDataFrame(recon_rows) \
    .withColumn("business_date", current_date()) \
    .withColumn("created_timestamp", current_timestamp()) \
    .withColumn("ai_summary", lit(None).cast("string")) \
    .withColumn("ai_recommended_action", lit(None).cast("string")) \
    .withColumn("ai_failure_category", lit(None).cast("string")) \
    .withColumn("ai_confidence_score", lit(None).cast("double")) \
    .withColumn("ai_model_version", lit(None).cast("string")) \
    .withColumn("ai_invocation_id", lit(None).cast("string")) \
    .withColumn("ai_enriched_timestamp", lit(None).cast("timestamp")) \
    .withColumn("agent1_failure_category", lit(None).cast("string"))

recon_df.write.mode("append").saveAsTable(RECON_RESULTS)

display(spark.sql(f"""
SELECT * FROM {RECON_RESULTS}
WHERE reconciliation_run_id = '{run_id}'
"""))